In [ ]:
## Benchmark Forecasting

# We evaluate benchmark models separately for each store.
# Each store uses its own history to create 42-day forecasts, and the metrics are stored per store and per model.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pmdarima import auto_arima
except ImportError:
    auto_arima = None

# ==========================================================
# Load data
# ==========================================================
processed_dir = Path("data/processed")
if not processed_dir.exists():
    processed_dir = Path.cwd().parent / "data" / "processed"

sales = pd.read_csv(processed_dir / "sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.sort_values(["store_id", "date"])

# ==========================================================
# Helper functions
# ==========================================================

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def build_forecasts(train_series, test_len):
    forecasts = {}

    train_series = pd.Series(train_series).astype(float)
    train_series = train_series.replace([np.inf, -np.inf], np.nan).dropna()

    if len(train_series) < 10:
        return {
            "Mean": np.repeat(np.nan, test_len),
            "Naive": np.repeat(np.nan, test_len),
            "Drift": np.repeat(np.nan, test_len),
            "Seasonal Naive": np.repeat(np.nan, test_len),
            "AutoARIMA": np.repeat(np.nan, test_len),
        }

    forecasts["Mean"] = np.full(test_len, train_series.mean())
    forecasts["Naive"] = np.full(test_len, train_series.iloc[-1])

    drift = np.arange(1, test_len + 1)
    forecasts["Drift"] = train_series.iloc[-1] + drift * (
        (train_series.iloc[-1] - train_series.iloc[0]) / (len(train_series) - 1)
    )

    forecasts["Seasonal Naive"] = np.array([
        train_series.iloc[-7 + (i % 7)] for i in range(test_len)
    ])

    if auto_arima is not None:
        try:
            model = auto_arima(
                train_series,
                seasonal=True,
                m=7,
                stepwise=True,
                suppress_warnings=True,
                error_action="ignore",
            )
            forecasts["AutoARIMA"] = model.predict(n_periods=test_len)
        except Exception:
            forecasts["AutoARIMA"] = np.repeat(np.nan, test_len)
    else:
        forecasts["AutoARIMA"] = np.repeat(np.nan, test_len)

    return forecasts

# ==========================================================
# Train / test split by store
# ==========================================================
# We use all rows before the 42-day holdout period as training.
# The forecast origin is the day before the first test date, so no extra day is removed.

forecast_horizon = 42
results = []

for store_id, store_df in sales.groupby("store_id"):
    store_df = store_df.sort_values("date").copy()
    store_df["sales"] = pd.to_numeric(store_df["sales"], errors="coerce")
    store_df = store_df.dropna(subset=["sales"])

    if len(store_df) < forecast_horizon + 5:
        continue

    train_series = store_df.iloc[:-forecast_horizon]["sales"].astype(float)
    test_series = store_df.iloc[-forecast_horizon:]["sales"].astype(float)
    test_dates = store_df.iloc[-forecast_horizon:]["date"]

    forecasts = build_forecasts(train_series, forecast_horizon)

    for model_name, pred in forecasts.items():
        pred = np.asarray(pred, dtype=float)
        if np.isnan(pred).any():
            continue

        results.append({
            "store_id": store_id,
            "model": model_name,
            "mae": mae(test_series.values, pred),
            "mape": mape(test_series.values, pred),
            "forecast_dates": json.dumps([d.strftime("%Y-%m-%d") for d in test_dates]),
            "actual_values": json.dumps([float(v) for v in test_series.values]),
            "forecast_values": json.dumps([float(v) for v in pred]),
        })

results_df = pd.DataFrame(results)
print(results_df.head())
print(f"\nTotal rows: {len(results_df)}")

# ==========================================================
# Save results
# ==========================================================
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "benchmark_results_per_store.csv"
results_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

# ==========================================================
# Summary by model
# ==========================================================
summary_df = (
    results_df.groupby("model")[["mae", "mape"]]
    .mean()
    .sort_values("mae")
)
print(summary_df)


In [ ]:
#The poor performance of the Naive and Drift baselines is mainly due to the data pattern around the test split. 
# The last training point is unusually low, while the first test point rises sharply, 
# so these simple methods struggle to adapt and produce large errors. 
# This is a characteristic of the series rather than a coding issue.